In [ ]:
# Install all required packages
!pip install -q torch torchvision scikit-learn scikit-image opencv-python tqdm pandas pillow numpy efficientnet_pytorch

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'CUDA version:    {torch.version.cuda}')
else:
    print('⚠️  No GPU detected. Training will be slow. Enable GPU in Runtime settings.')

print('\n✓ Setup complete!')

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

# ── Detect environment ────────────────────────────────────────────────────────
IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB  = Path('/content').exists() and not IS_KAGGLE
WORK_BASE = Path('/kaggle/working') if IS_KAGGLE else Path('/content')

REPO_URL  = 'https://github.com/Akshit0707/Cerivcal-Cancer-Stage-Classification'
REPO_DIR  = WORK_BASE / 'repo'

print(f'Environment: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')
print(f'Working dir: {WORK_BASE}')

# ── Clone or update repo ──────────────────────────────────────────────────────
if REPO_DIR.exists():
    print(f'Repo already exists at {REPO_DIR}, pulling latest...')
    result = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print(f'Cloning {REPO_URL} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('Git clone failed. Check the repo URL and network access.')
    print(result.stdout)

# ── Confirm structure ─────────────────────────────────────────────────────────
print('\nRepo contents:')
for p in sorted(REPO_DIR.iterdir()):
    print(f'  {p.name}/'  if p.is_dir() else f'  {p.name}')

print(f'\n✓ Repo ready at: {REPO_DIR}')

In [ ]:
# ── Find train_hybrid.py inside the cloned repo ───────────────────────────────
scripts = list(REPO_DIR.rglob('train_hybrid.py'))
if not scripts:
    raise FileNotFoundError(
        'train_hybrid.py not found in repo. '
        'Ensure the file exists at backend/scripts/train_hybrid.py in the repo.'
    )

SCRIPT_PATH = scripts[0]
SCRIPT_DIR  = SCRIPT_PATH.parent

# ── Determine repo root (2 levels up from scripts/) ──────────────────────────
# Expected layout: <repo_root>/backend/scripts/train_hybrid.py
if (SCRIPT_PATH.parents[1] / 'backend').exists():
    REPO_ROOT = SCRIPT_PATH.parents[2]
elif (SCRIPT_PATH.parent / '__init__.py').exists() or (SCRIPT_PATH.parents[1]).name == 'backend':
    REPO_ROOT = SCRIPT_PATH.parents[2]
else:
    REPO_ROOT = SCRIPT_PATH.parent

BACKEND_DIR = REPO_ROOT / 'backend'

# ── Inject all needed paths into sys.path ────────────────────────────────────
for p in [str(REPO_ROOT), str(BACKEND_DIR), str(SCRIPT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✓ Script found : {SCRIPT_PATH}')
print(f'✓ Script dir   : {SCRIPT_DIR}')
print(f'✓ Repo root    : {REPO_ROOT}')
print(f'✓ Backend dir  : {BACKEND_DIR}')
print(f'✓ sys.path     : {sys.path[:4]}')
print('\n✓ Path setup complete!')

In [ ]:
import os

LOCAL_DATA = WORK_BASE / 'data'

def find_dataset_root():
    """Search common mount points for a dataset with train/ and val/ folders."""
    roots = [
        Path('/kaggle/input'),
        Path('/kaggle/working'),
        Path('/content/drive/MyDrive'),
        Path('/content'),
        REPO_DIR / 'data',
        Path('.'),
    ]
    for root in roots:
        if not root.exists():
            continue
        for train_dir in root.rglob('train'):
            if train_dir.is_dir() and (train_dir.parent / 'val').is_dir():
                return train_dir.parent
    return None

# ── Copy dataset to working dir if needed ────────────────────────────────────
if LOCAL_DATA.exists() and (LOCAL_DATA / 'train').exists() and (LOCAL_DATA / 'val').exists():
    print(f'✅ Data already at {LOCAL_DATA}')
    DATA_DIR = LOCAL_DATA
else:
    found = find_dataset_root()
    if found is None:
        raise FileNotFoundError(
            '❌ No dataset with train/ and val/ folders found.\n'
            'On Kaggle: add your dataset via "Add Data" → attach cervical cancer dataset.\n'
            'On Colab:  upload your data or mount Google Drive.'
        )
    print(f'✅ Found dataset at: {found}')
    if not found.samefile(LOCAL_DATA) if LOCAL_DATA.exists() else True:
        print('📦 Copying to working directory...')
        if LOCAL_DATA.exists():
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(str(found), str(LOCAL_DATA))
        print('✅ Data copied!')
    DATA_DIR = LOCAL_DATA

# ── Count images per split/class ─────────────────────────────────────────────
def count_images(directory):
    classes = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            classes[class_name] = count
    return classes

print('\n📊 Data structure:')
total_train, total_val = 0, 0
for split in ['train', 'val']:
    split_dir = DATA_DIR / split
    if split_dir.exists():
        counts = count_images(split_dir)
        print(f'\n  {split.upper()}:')
        for cls, n in counts.items():
            print(f'    {cls:25s}: {n:4d} images')
        total = sum(counts.values())
        print(f'    {"TOTAL":25s}: {total:4d}')
        if split == 'train': total_train = total
        else: total_val = total

print(f'\n  Train/Val split: {total_train}/{total_val}')
print(f'\n✓ Dataset ready at: {DATA_DIR}')

In [ ]:
import os

CHECKPOINT_DIR = WORK_BASE / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU-tuned hyperparameters ─────────────────────────────────────────────────
# These settings are tuned for 80–90% accuracy with EfficientNet-B3
EPOCHS        = 100        # early stopping will kick in ~epoch 40–60
BATCH_SIZE    = 32         # increase to 64 if GPU has ≥16 GB VRAM
LEARNING_RATE = 1e-4       # backbone gets 1e-5 automatically (×0.1)
PATIENCE      = 20         # early stopping patience
NUM_WORKERS   = 4          # parallel data loading on GPU

# ── Build PYTHONPATH for subprocess ──────────────────────────────────────────
extra_paths = [
    str(REPO_ROOT),
    str(BACKEND_DIR),
    str(SCRIPT_DIR),
]
env = os.environ.copy()
existing_pp = env.get('PYTHONPATH', '')
env['PYTHONPATH'] = ':'.join(extra_paths) + (':' + existing_pp if existing_pp else '')
env['PYTHONUNBUFFERED'] = '1'

# ── Build CLI command ─────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(SCRIPT_PATH),
    '--data-dir',                str(DATA_DIR),
    '--epochs',                  str(EPOCHS),
    '--batch-size',              str(BATCH_SIZE),
    '--learning-rate',           str(LEARNING_RATE),
    '--early-stopping-patience', str(PATIENCE),
    '--num-workers',             str(NUM_WORKERS),
    '--checkpoint-dir',          str(CHECKPOINT_DIR),
    '--seed',                    '42',
    # Remove --cpu-only flag → GPU will be auto-detected
]

print('=' * 70)
print('TRAINING CONFIGURATION')
print('=' * 70)
print(f'  Script         : {SCRIPT_PATH}')
print(f'  Data dir       : {DATA_DIR}')
print(f'  Checkpoint dir : {CHECKPOINT_DIR}')
print(f'  Epochs         : {EPOCHS} (early stop patience={PATIENCE})')
print(f'  Batch size     : {BATCH_SIZE}')
print(f'  Learning rate  : {LEARNING_RATE} (backbone gets {LEARNING_RATE*0.1:.1e})')
print(f'  Workers        : {NUM_WORKERS}')
print(f'  GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}') 
print(f'  AMP            : {"Enabled" if torch.cuda.is_available() else "Disabled"}')
print('=' * 70)
print('\nStarting training...\n')

# ── Run training subprocess ───────────────────────────────────────────────────
result = subprocess.run(
    cmd,
    cwd=str(REPO_ROOT),
    env=env,
    check=True
)
print('\n✓ Training complete!')

In [ ]:
import sys, os, subprocess, shutil
from pathlib import Path

def find_train_script_in_roots(roots):
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("backend/scripts/train_hybrid.py"):
            return p
        for p in root.rglob("train_hybrid.py"):
            return p
    return None

search_roots = [Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/working"), Path("/content"), Path(".")]
input_script = find_train_script_in_roots(search_roots)
if input_script is None:
    raise FileNotFoundError("train_hybrid.py not found in inputs or repo.")

if (input_script.parents[2] / "backend").exists():
    SRC_ROOT = input_script.parents[2]
else:
    SRC_ROOT = input_script.parent.parent

WORK_DIR = Path("/kaggle/working/repo")
if WORK_DIR.exists():
    try:
        if not SRC_ROOT.samefile(WORK_DIR):
            shutil.rmtree(WORK_DIR)
            shutil.copytree(SRC_ROOT, WORK_DIR)
    except Exception:
        shutil.rmtree(WORK_DIR)
        shutil.copytree(SRC_ROOT, WORK_DIR)
else:
    shutil.copytree(SRC_ROOT, WORK_DIR)

candidate = WORK_DIR / "backend" / "scripts" / "train_hybrid.py"
script_path = candidate if candidate.exists() else next(WORK_DIR.rglob("train_hybrid.py"))

repo_root = WORK_DIR
script_dir = script_path.parent

# ── AUTO-FIND models/ package anywhere under /kaggle/input ──
def find_models_parent(base: Path):
    for p in base.rglob("models/__init__.py"):
        return p.parent.parent  # parent of models/ folder
    return None

models_parent = find_models_parent(Path("/kaggle/input"))
if models_parent is None:
    raise FileNotFoundError("Could not find models/__init__.py under /kaggle/input. Check your dataset is added.")

print(f"models package found at: {models_parent / 'models'}")

data_dir = Path("/kaggle/working/data").resolve()
checkpoint_dir = Path("/kaggle/working/checkpoints").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

EPOCHS = 100
BATCH_SIZE = 16  # smaller batch for better gradient updates on small classes
LEARNING_RATE = 5e-5  # even lower LR
PATIENCE = 15
NUM_WORKERS = 2

env = os.environ.copy()
extra_paths = [
    str(models_parent),
    str(script_dir),
    str(repo_root),
    str(repo_root / "backend"),
]
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = ":".join(extra_paths) + (":" + existing if existing else "")

# Ensure pip packages available in subprocess
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable, str(script_path),
    "--data-dir", str(data_dir),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--early-stopping-patience", str(PATIENCE),
    "--num-workers", str(NUM_WORKERS),
    "--checkpoint-dir", str(checkpoint_dir),
]

print("Using script:        ", script_path)
print("Using data dir:      ", data_dir)
print("Using checkpoint dir:", checkpoint_dir)
print("PYTHONPATH:          ", env["PYTHONPATH"])
print("Running:", " ".join(cmd))

subprocess.run(cmd, cwd=str(repo_root), check=True, env=env)

In [ ]:
import zipfile

# ── Check saved checkpoints ───────────────────────────────────────────────────
print('📁 Saved checkpoints:')
best_path = None
for p in sorted(CHECKPOINT_DIR.rglob('*.pth')):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:40s}  {size_mb:.1f} MB')
    if 'best' in p.name:
        best_path = p

# ── Load and print best checkpoint metadata ───────────────────────────────────
if best_path:
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    print(f'\n🏆 Best Model Summary:')
    print(f'  Epoch          : {ckpt.get("epoch", "N/A")}')
    print(f'  Val Accuracy   : {ckpt.get("val_accuracy", 0):.2f}%')
    print(f'  Val Loss       : {ckpt.get("val_loss", 0):.4f}')
    print(f'  Num Classes    : {ckpt.get("num_classes", "N/A")}')
    print(f'  Class Mapping  : {ckpt.get("class_to_idx", {})}')
    attn = ckpt.get('avg_attention_weights')
    if attn is not None and len(attn) == 2:
        import numpy as np
        print(f'  CNN branch weight  : {float(attn[0]):.3f}')
        print(f'  Feature branch wt  : {float(attn[1]):.3f}')
else:
    print('\n⚠️  No best_hybrid_model.pth found. Check training logs for errors.')

# ── Zip for download ──────────────────────────────────────────────────────────
zip_path = WORK_BASE / 'trained_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    if best_path and best_path.exists():
        zf.write(best_path, arcname=best_path.name)
    for p in sorted(CHECKPOINT_DIR.rglob('*.pth')):
        if p != best_path:
            zf.write(p, arcname=p.relative_to(CHECKPOINT_DIR))

print(f'\n✓ Zip created: {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)')

# ── Auto-download (Colab) / instructions (Kaggle) ─────────────────────────────
try:
    from google.colab import files
    files.download(str(zip_path))
    print('✓ Download started (Colab).')
except ImportError:
    print('ℹ️  On Kaggle: go to Output panel → download trained_model.zip')

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import numpy as np

# ── Re-import model class from repo ──────────────────────────────────────────
# sys.path already set up in Step 2
try:
    from backend.feature_extractor import extract_medical_features
except ImportError:
    from feature_extractor import extract_medical_features

# Import EfficientNetHybrid directly from the script (not the old stub)
import importlib.util
spec = importlib.util.spec_from_file_location('train_hybrid', str(SCRIPT_PATH))
train_hybrid_mod = importlib.util.load_from_spec(spec)
spec.loader.exec_module(train_hybrid_mod)
EfficientNetHybrid = train_hybrid_mod.EfficientNetHybrid

# ── Load checkpoint ───────────────────────────────────────────────────────────
if not best_path or not best_path.exists():
    print('No best checkpoint found. Run Step 4 first.')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    num_classes  = ckpt['num_classes']
    num_features = ckpt.get('num_traditional_features', 30)
    class_to_idx = ckpt['class_to_idx']
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    model = EfficientNetHybrid(num_classes=num_classes, num_features=num_features)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    print(f'✓ Model loaded from: {best_path}')
    print(f'  Classes: {idx_to_class}')

    # ── Pick a sample image from val set ─────────────────────────────────────
    val_dir = DATA_DIR / 'val'
    sample_img_path = None
    for cls_dir in val_dir.iterdir():
        if cls_dir.is_dir():
            imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
            if imgs:
                sample_img_path = imgs[0]
                true_label = cls_dir.name
                break

    if sample_img_path:
        img = Image.open(sample_img_path).convert('RGB')

        # Preprocess
        tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        img_tensor = tf(img).unsqueeze(0).to(device)

        raw_feats = extract_medical_features(img)
        feat_tensor = torch.tensor(raw_feats, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits, attn = model(img_tensor, feat_tensor)
            probs = F.softmax(logits, dim=-1)[0]
            pred_idx = probs.argmax().item()

        print(f'\n🔍 Inference on: {sample_img_path.name}')
        print(f'  True label  : {true_label}')
        print(f'  Predicted   : {idx_to_class[pred_idx]} (conf: {probs[pred_idx]*100:.1f}%)')
        print(f'  All probs   : { {idx_to_class[i]: f"{p*100:.1f}%" for i, p in enumerate(probs.cpu().numpy())} }')
        print(f'  Attn (CNN/Feat): {float(attn[0,0]):.3f} / {float(attn[0,1]):.3f}')
    else:
        print('No sample images found in val directory.')